# 01: Generowanie zbioru syntetycznego

**Wynik:** `MyDrive/rs_project/datasets/rs_dataset_256.zip` (+ `rs_dataset_256_config.json` z parametrami generatora).
Uruchamiasz tylko wtedy, gdy zmieniają się parametry kamer (`configs/config_webcam.json`) albo generator.
Przy zmianie parametrów nadaj nową nazwę (`NAME`), żeby nie nadpisać zbioru, na którym trenowano wcześniejsze modele.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
NAME = 'rs_dataset_256_full'              # 16k par; przy nowych parametrach kamer nadaj nową nazwę, np. '..._v2'
CONFIG = 'configs/config_webcam.json'
PER_IMAGE, SIZE = 20, 256
existing = P.find_dataset_zip(NAME)
print('istnieje już:' if existing else 'brak, generuję:', existing or P.DATASETS / f'{NAME}.zip')

## 1. Zdjęcia źródłowe: DIV2K (800 ostrych zdjęć 2K, ok. 3,5 GB)

In [ ]:
!wget -q -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -O /content/div2k.zip
!unzip -q -o /content/div2k.zip -d /content/div2k && ls /content/div2k/DIV2K_train_HR | wc -l

## 2. Generowanie (800 zdjęć × 20 = 16 tys. par)

In [ ]:
!rm -rf /content/dataset
!python src/synth_rs.py --src /content/div2k/DIV2K_train_HR --out /content/dataset \
    --per-image {PER_IMAGE} --size {SIZE} --workers {os.cpu_count()} --config {CONFIG}
!python src/preview.py /content/dataset/train --n 4 --out /content/preview.png
from IPython.display import Image; Image('/content/preview.png')

## 3. Zapis na Drive

In [ ]:
import shutil
!cd /content && zip -qr /content/{NAME}.zip dataset
shutil.copy(f'/content/{NAME}.zip', P.DATASETS / f'{NAME}.zip')
shutil.copy('/content/dataset/config.json', P.DATASETS / f'{NAME}_config.json')
P.status()